# BellAI - Plataforma Modular de Entrenamiento en Google Colab

Este cuaderno esta dividido en celdas independientes:
- **Parte 1**: Instalacion y Configuracion (Se ejecuta solo una vez al inicio de sesion).
- **Partes 2 a 6**: Celdas dedicadas para cada uno de los 5 modelos de BellAI con formularios visuales (`#@param`). Solo eliges el modelo deseado y pulsas Play.

## Parte 1: Instalacion y Configuracion del Entorno
Monta Google Drive, configura credenciales de Kaggle, clona/actualiza el repositorio e instala dependencias.

In [ ]:
#@title 1. Inicializar Entorno BellAI { display-mode: "form" }
import os
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Token de Kaggle (Reemplaza con tu token privado)
KAGGLE_API_TOKEN = 'TU_KAGGLE_API_TOKEN_AQUI' #@param {type:"string"}
if KAGGLE_API_TOKEN and KAGGLE_API_TOKEN != 'TU_KAGGLE_API_TOKEN_AQUI':
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN
    !mkdir -p ~/.kaggle && echo $KAGGLE_API_TOKEN > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
    print('[*] Kaggle API Token configurado exitosamente.')

# 3. Clonar o actualizar repositorio
if not os.path.exists('/content/modelIATrainer'):
    !git clone https://github.com/ldgd2/modelIATrainer.git
else:
    %cd /content/modelIATrainer
    !git pull

%cd /content/modelIATrainer
!pip install -q -r requirements.txt

print('\n' + '='*55)
print(' [*] ENTORNO BELLAI CONFIGURADO Y LISTO')
print('='*55)


## Parte 2: Modulo 1 - Armario Digital
Deteccion y segmentacion de prendas con DeepFashion2 hacia `wardrobe_v1.pt`.

In [ ]:
#@title Entrenar Armario Digital { display-mode: "form" }
%cd /content/modelIATrainer

epocas = 10 #@param {type:"integer"}
batch_size = 32 #@param {type:"integer"}
fuente_datos = "Simulacion / Verificacion" #@param ["Descargar con Kagglehub", "Enlaces directos wget", "Simulacion / Verificacion"]
link_train = "" #@param {type:"string"}
link_val = "" #@param {type:"string"}
password = "2019Deepfashion2**" #@param {type:"string"}
resume_checkpoint = "" #@param {type:"string"}

args_datos = ""
if fuente_datos == "Descargar con Kagglehub":
    args_datos = "--download_data"
elif fuente_datos == "Enlaces directos wget":
    if link_train:
        args_datos += f' --link_train "{link_train}"'
    if link_val:
        args_datos += f' --link_val "{link_val}"'

args_resume = f'--resume "{resume_checkpoint}"' if resume_checkpoint else ""

cmd = f'python train.py --module armario_digital --epochs {epocas} --save_every 2 --batch_size {batch_size} --password "{password}" {args_datos} {args_resume}'.strip()
print(f'[*] Ejecutando: {cmd}\n')
!{cmd}


## Parte 3: Modulo 2 - Diagnostico Morfologico
Analisis de rostro, silueta corporal y cabello (FairFace, COCO Keypoints, CelebAMask-HQ).

In [ ]:
#@title Entrenar Diagnostico Morfologico { display-mode: "form" }
%cd /content/modelIATrainer

subtarea = "colorimetria" #@param ["colorimetria", "morfologia", "capilar"]
epocas = 10 #@param {type:"integer"}
batch_size = 16 #@param {type:"integer"}
descargar_datos = True #@param {type:"boolean"}
resume_checkpoint = "" #@param {type:"string"}

dl_flag = "--download_data" if descargar_datos else ""
args_resume = f'--resume "{resume_checkpoint}"' if resume_checkpoint else ""

cmd = f'python train.py --module diagnostico_morfologico --task {subtarea} --epochs {epocas} --save_every 2 --batch_size {batch_size} {dl_flag} {args_resume}'.strip()
print(f'[*] Ejecutando: {cmd}\n')
!{cmd}


## Parte 4: Modulo 3 - Motor de Recomendacion
Compatibilidad estetica de conjuntos de ropa (PolyVore -> `style_recommender_v1.pt`).

In [ ]:
#@title Entrenar Motor de Recomendacion { display-mode: "form" }
%cd /content/modelIATrainer

epocas = 10 #@param {type:"integer"}
batch_size = 32 #@param {type:"integer"}
link_polyvore = "" #@param {type:"string"}
resume_checkpoint = "" #@param {type:"string"}

args_link = f'--link_polyvore "{link_polyvore}"' if link_polyvore else ""
args_resume = f'--resume "{resume_checkpoint}"' if resume_checkpoint else ""

cmd = f'python train.py --module motor_recomendacion --epochs {epocas} --save_every 2 --batch_size {batch_size} {args_link} {args_resume}'.strip()
print(f'[*] Ejecutando: {cmd}\n')
!{cmd}


## Parte 5: Modulo 4 - Simulacion Virtual
Probador virtual y mapeo de maquillaje (VITON-HD, MT-Dataset).

In [ ]:
#@title Entrenar Simulacion Virtual { display-mode: "form" }
%cd /content/modelIATrainer

subtarea = "probador" #@param ["probador", "maquillaje"]
epocas = 10 #@param {type:"integer"}
batch_size = 8 #@param {type:"integer"}
descargar_datos = True #@param {type:"boolean"}
resume_checkpoint = "" #@param {type:"string"}

dl_flag = "--download_data" if descargar_datos else ""
args_resume = f'--resume "{resume_checkpoint}"' if resume_checkpoint else ""

cmd = f'python train.py --module simulacion_virtual --task {subtarea} --epochs {epocas} --save_every 2 --batch_size {batch_size} {dl_flag} {args_resume}'.strip()
print(f'[*] Ejecutando: {cmd}\n')
!{cmd}


## Parte 6: Modulo 5 - Asistente Conversacional
Descarga y empaquetado del modelo LLM local en formato GGUF desde Hugging Face.

In [ ]:
#@title Descargar Modelo GGUF { display-mode: "form" }
%cd /content/modelIATrainer

repo_id = "TheBloke/Llama-2-7B-Chat-GGUF" #@param {type:"string"}
filename = "bellai_assistant_8b.gguf" #@param {type:"string"}

cmd = f'python train.py --module asistente_conversacional --repo_id "{repo_id}" --filename "{filename}"'
print(f'[*] Ejecutando: {cmd}\n')
!{cmd}
